# 09 — Final Radiomics Pipeline (Scientific Reports Revision)

This notebook implements the **corrected** GLOO (Leave-One-Mouse-Out) cross-validation pipeline for predicting cure vs. relapse in GL261 glioblastoma mice using PyRadiomics `original_` features and XGBoost.

## Corrections vs. Previous Version

| Item | Change |
|------|--------|
| **A1** | Feature selection moved **inside** each GLOO fold (training data only — no data leakage) |
| **A2** | Animal-level majority-vote metrics added |
| **A3** | Stratified temporal analysis (early/mid/late terciles by day_of_study) |
| **A4** | Bootstrap 95% CIs (10,000 samples) on exam-level metrics |

## Dataset
- 10 treated GL261 mice: 5 cured (label=1), 5 relapsing (label=0)
- 298 MRI exams total
- Features: PyRadiomics `original_` texture/shape/first-order features
- Cross-validation: GLOO (Leave-One-Mouse-Out, 10 folds)


## Setup

Import all required libraries. `neuroHarmonize` is **not** used (removed per revision plan). Lag features (delta_2, rolling_mean_3, slope_3) are **not** computed — not part of the paper's final pipeline.


In [1]:
import os
import sys

# Set working directory to repo root so config and src are importable.
# The kernel may start from a different directory when run via nbconvert.
_candidate = os.path.abspath('.')
if not os.path.isfile(os.path.join(_candidate, 'config.py')):
    # Walk up to find the repo root (contains config.py)
    _search = _candidate
    for _ in range(5):
        _search = os.path.dirname(_search)
        if os.path.isfile(os.path.join(_search, 'config.py')):
            _candidate = _search
            break
os.chdir(_candidate)
sys.path.insert(0, _candidate)

import warnings

warnings.filterwarnings('ignore')

from collections import defaultdict

import numpy as np
import pandas as pd
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from scipy.stats import mannwhitneyu
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler

import config
from src.datasplitting import DataSplitting
from src.processing import Processing

print('XGBoost version:', xgb.__version__)
print('Working directory:', os.getcwd())


XGBoost version: 3.2.0
Working directory: C:\Users\U1660450\OneDrive - Grupo Caixabank\Proyectos\ML-Strategies-Inflection-Point-GL261-Response-Relapse


## Configuration

Hyperparameters from Table S3 of the paper. XGBoost params use clean keys (no `xgbclassifier__` prefix). Output paths use `config.OUTPUT_EXP_09`.


In [2]:
SEED = 42
SAMPLE_SIZE = 10   # one fold per mouse (10 treated mice)
N_BOOTSTRAP = 10_000
np.random.seed(SEED)

# XGBoost hyperparameters (Table S3) — clean keys, no pipeline prefix, no use_label_encoder
XGB_PARAMS = {
    'colsample_bytree': 0.6,
    'gamma': 0.5,
    'learning_rate': 0.01,
    'max_depth': 3,
    'min_child_weight': 1,
    'n_estimators': 100,
    'subsample': 0.6,
    'random_state': SEED,
}

SPLIT_METHOD = 'grouped_combinations_split_with_sampling'
OUTPUT_DIR = config.OUTPUT_EXP_09

os.makedirs(os.path.join(OUTPUT_DIR, SPLIT_METHOD), exist_ok=True)
print('Output directory:', OUTPUT_DIR)
print('XGB params:', XGB_PARAMS)


Output directory: outputs\paper_revision\radiomics\09_final_pipeline
XGB params: {'colsample_bytree': 0.6, 'gamma': 0.5, 'learning_rate': 0.01, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 100, 'subsample': 0.6, 'random_state': 42}


## Data Loading

Load radiomics features from `config.RADIOMICS_FINAL_FEATURES_NEW_FIXED`. Remove the 8 control (non-treated) mice, leaving 10 treated mice (5 cured, 5 relapsing) and 298 exams.

**Control mice removed:** 1258, 1260, 1261, 1297, 1299, 1359, 1360, 1361


In [3]:
dataclass = Processing(
    source_path=config.RADIOMICS_FINAL_FEATURES_NEW_FIXED,
    all_features=False
)
dataclass.read()
df = dataclass.df.copy()

# Remove control (non-treated) mice
CONTROL_MICE = [1258, 1260, 1261, 1297, 1299, 1359, 1360, 1361]
for m in CONTROL_MICE:
    df = df[df['m_id'] != m]

df = df.reset_index(drop=True)
DS_LENGTH = len(df)

print('Dataset shape:', df.shape)
print('Unique treated mice:', sorted(df['m_id'].unique()))
print('\nLabel distribution (group_name):')
print(df.groupby('group_name')['m_id'].nunique().rename('n_mice'))
print('\nExams per mouse:')
print(df.groupby(['m_id', 'group_name']).size().reset_index(name='n_exams').to_string(index=False))


Dataset shape: (298, 1221)
Unique treated mice: [np.int64(1263), np.int64(1264), np.int64(1270), np.int64(1276), np.int64(1281), np.int64(1284), np.int64(1285), np.int64(1380), np.int64(1382), np.int64(1383)]

Label distribution (group_name):
group_name
0    5
1    5
Name: n_mice, dtype: int64

Exams per mouse:
 m_id  group_name  n_exams
 1263           0       56
 1264           0       43
 1270           0       43
 1276           1       21
 1281           1       12
 1284           1        8
 1285           1       17
 1380           0       47
 1382           1        8
 1383           0       43


## Feature Selection Inside Each GLOO Fold (A1)

**What:** Feature selection (variance filter + Spearman correlation deduplication + Mann-Whitney U test + `original_`-only heuristic) is applied **inside** each GLOO fold, fitted exclusively on training subjects.

**Why:** Running feature selection on the full dataset before splitting leaks information from the held-out test mouse into the selection step — the variance and correlation structure of the test exams influences which features are retained. By refitting selection within each fold, the test mouse contributes zero information to feature selection.

> Feature selection is refitted within each GLOO fold using training subjects only, preventing data leakage from test subjects into the feature selection step (addressing Reviewer Comment 1).

### Steps per fold
1. **Heuristic pre-filter** — keep only `original_` features (applied first for efficiency; non-original features are dropped regardless, and applying this first prevents them from removing original features via Spearman deduplication)
2. **Variance filter** — drop features with variance < 0.01 on training data
3. **Spearman deduplication** — for each pair with |rho| > 0.95, drop the later feature
4. **Mann-Whitney U** — keep features with p <= 0.05 between classes (two-sided) on training data


In [4]:
def select_features_on_train(X_train, y_train):
    '''
    Perform all feature selection steps using TRAINING data only.
    No information from the test mouse leaks into this process.

    Steps (heuristic applied first for computational efficiency):
        0. Heuristic: keep only original_ features
        1. Variance filter: var >= 0.01
        2. Spearman deduplication: drop one of each correlated pair (|rho| > 0.95)
        3. Mann-Whitney U: keep p <= 0.05 (two-sided)

    Args:
        X_train (pd.DataFrame): training features (may include m_id, day_of_study)
        y_train (pd.Series): training labels (0=relapse, 1=cure)

    Returns:
        list: selected feature column names (excludes m_id, day_of_study)
    '''
    # Exclude metadata columns
    feature_cols = [c for c in X_train.columns if c not in ('m_id', 'day_of_study')]
    selected = feature_cols[:]

    # Step 0: Heuristic — keep only original_ features (applied first for efficiency).
    # Reduces ~1218 features to ~107 before expensive Spearman computation.
    # Non-original_ features are removed in any case; applying this first ensures
    # they do not cause original_ features to be dropped in the Spearman step.
    filter_prefixes = ('wavelet', 'log-sigma', 'square_', 'squareroot', 'logarithm', 'exponential', 'gradient')
    selected = [f for f in selected if not any(f.startswith(p) for p in filter_prefixes)]

    # Step 1: Variance filter
    if not selected:
        return selected
    variances = X_train[selected].var()
    selected = [f for f in selected if variances[f] >= 0.01]

    # Step 2: Spearman correlation deduplication
    if len(selected) > 1:
        corr = X_train[selected].corr(method='spearman').abs()
        to_drop = set()
        for i in range(len(selected)):
            for j in range(i + 1, len(selected)):
                if selected[j] not in to_drop and corr.iloc[i, j] > 0.95:
                    to_drop.add(selected[j])
        selected = [f for f in selected if f not in to_drop]

    # Step 3: Mann-Whitney U test
    mw_keep = []
    for feat in selected:
        g0 = X_train.loc[y_train == 0, feat].dropna()
        g1 = X_train.loc[y_train == 1, feat].dropna()
        if len(g0) < 2 or len(g1) < 2:
            continue
        _, p = mannwhitneyu(g0, g1, alternative='two-sided')
        if p <= 0.05:
            mw_keep.append(feat)
    selected = mw_keep

    return selected

print('Feature selection function defined.')
print('Called INSIDE each GLOO fold on training data only (A1 fix).')


Feature selection function defined.
Called INSIDE each GLOO fold on training data only (A1 fix).


## GLOO Cross-Validation Loop

For each of the 10 folds (one mouse left out as test):
1. Feature selection runs on training data only (A1)
2. Pipeline: `SimpleImputer(mean)` → `StandardScaler` → `SMOTE(k=5)` → `XGBClassifier`
3. Predictions (y_true, y_pred, y_proba, m_id, day_of_study) are pooled across all folds
4. Per-fold `predictions.csv` is saved

Note: `day_of_study` and `m_id` are excluded from model input but retained for tracking.


In [5]:
# Generate GLOO splits: 10 folds, one per mouse
split_method, m_data = DataSplitting.grouped_combinations_split_with_sampling(
    df=df, seed=SEED, sample_size=SAMPLE_SIZE
)

# Pooled collectors (across all folds)
all_y_true = []
all_y_pred = []
all_y_proba = []
all_mouse_ids = []
all_days = []
all_n_features = []

# Feature importance accumulator: feature_name -> list of importances per fold
feature_importance_accum = defaultdict(list)

print(f'Running {len(m_data)} GLOO folds ...')
print('=' * 60)

for fold_idx, fold_split in m_data.items():
    X_train_raw = fold_split['X_train']
    y_train = fold_split['y_train']
    X_test_raw = fold_split['X_test']
    y_test = fold_split['y_test']

    # Extract metadata BEFORE feature selection / model training
    test_mouse = X_test_raw['m_id'].unique()
    m_id_test = X_test_raw['m_id'].values.copy()
    day_test = X_test_raw['day_of_study'].values.copy()
    y_test_vals = y_test.values

    print(f'\nFold {fold_idx}: test mouse = {test_mouse}, n_test_exams = {len(y_test_vals)}')

    # A1: Feature selection on TRAINING data only (no leakage from test mouse)
    selected = select_features_on_train(X_train_raw, y_train)
    n_sel = len(selected)

    if n_sel == 0:
        # Fallback: all original_ features if selection yields nothing
        selected = [
            c for c in X_train_raw.columns
            if c.startswith('original_') and c not in ('m_id', 'day_of_study')
        ]
        print(f'  WARNING: Fallback to {len(selected)} original_ features (selection returned 0)')
        n_sel = len(selected)
    else:
        print(f'  Selected {n_sel} features')

    # Prepare model inputs: only selected features, no metadata
    X_tr = X_train_raw[selected].copy()
    X_te = X_test_raw[selected].copy()

    # Pipeline: imputer -> scaler -> SMOTE -> XGBClassifier
    # Uses imblearn.pipeline.Pipeline so SMOTE is only applied during fit()
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('smote', SMOTE(k_neighbors=5, random_state=SEED)),
        ('clf', xgb.XGBClassifier(**XGB_PARAMS)),
    ])
    pipe.fit(X_tr, y_train)

    # Predictions
    y_pred = pipe.predict(X_te)
    y_proba = pipe.predict_proba(X_te)[:, 1]

    # Accumulate feature importances (keyed by feature name)
    xgb_imp = pipe.named_steps['clf'].feature_importances_
    for feat, imp in zip(selected, xgb_imp):
        feature_importance_accum[feat].append(float(imp))

    # Save per-fold predictions
    fold_dir = os.path.join(OUTPUT_DIR, SPLIT_METHOD, f'fold_{fold_idx}')
    os.makedirs(fold_dir, exist_ok=True)
    pd.DataFrame({
        'm_id': m_id_test,
        'day_of_study': day_test,
        'y_true': y_test_vals,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'n_features_selected': n_sel,
    }).to_csv(os.path.join(fold_dir, 'predictions.csv'), index=False)

    # Accumulate for pooled analysis
    all_y_true.extend(y_test_vals.tolist())
    all_y_pred.extend(y_pred.tolist())
    all_y_proba.extend(y_proba.tolist())
    all_mouse_ids.extend(m_id_test.tolist())
    all_days.extend(day_test.tolist())
    all_n_features.append(n_sel)

print('\n' + '=' * 60)
print(f'All {len(m_data)} folds complete. Pooled exams: {len(all_y_true)}')
print(f'Features selected per fold: mean={np.mean(all_n_features):.1f}, '
      f'std={np.std(all_n_features):.1f}, '
      f'min={min(all_n_features)}, max={max(all_n_features)}')

# Convert to arrays for downstream analysis
all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)
all_y_proba = np.array(all_y_proba)
all_mouse_ids = np.array(all_mouse_ids)
all_days = np.array(all_days)


(np.int64(1276),)
Total possible combinations: 10 of size 1
Number of sampled combinations: 10
Running 10 GLOO folds ...

Fold 0: test mouse = [1281], n_test_exams = 12
  Selected 18 features

Fold 1: test mouse = [1276], n_test_exams = 21
  Selected 25 features

Fold 2: test mouse = [1382], n_test_exams = 8
  Selected 23 features

Fold 3: test mouse = [1383], n_test_exams = 43
  Selected 21 features

Fold 4: test mouse = [1264], n_test_exams = 43
  Selected 22 features

Fold 5: test mouse = [1263], n_test_exams = 56
  Selected 22 features

Fold 6: test mouse = [1380], n_test_exams = 47
  Selected 23 features

Fold 7: test mouse = [1284], n_test_exams = 8
  Selected 21 features

Fold 8: test mouse = [1285], n_test_exams = 17
  Selected 22 features

Fold 9: test mouse = [1270], n_test_exams = 43
  Selected 20 features

All 10 folds complete. Pooled exams: 298
Features selected per fold: mean=21.7, std=1.8, min=18, max=25


## Exam-Level Results

Pool all fold predictions and compute metrics at the exam level (each MRI scan is one observation). Metrics: AUC, Sensitivity (Recall for class 1), Specificity (Recall for class 0), PPV (Precision), NPV, Accuracy.


In [6]:
def compute_metrics(y_true, y_pred, y_proba):
    '''
    Compute classification metrics from pooled predictions.
    Uses labels=[0,1] in confusion_matrix to ensure 2x2 matrix even
    when one class is absent from predictions.
    '''
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sens = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    spec = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    npv  = tn / (tn + fn) if (tn + fn) > 0 else float('nan')
    acc  = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else float('nan')

    try:
        auc_val = roc_auc_score(y_true, y_proba)
    except ValueError:
        auc_val = float('nan')

    return {
        'AUC': auc_val,
        'Sensitivity': sens,
        'Specificity': spec,
        'PPV': ppv,
        'NPV': npv,
        'Accuracy': acc,
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
    }


exam_metrics = compute_metrics(all_y_true, all_y_pred, all_y_proba)

print('Exam-Level Metrics (pooled across all 10 GLOO folds):')
print('-' * 45)
for k, v in exam_metrics.items():
    if isinstance(v, float):
        print(f'  {k:<14}: {v:.4f}')
    else:
        print(f'  {k:<14}: {v}')

# Save
exam_metrics_df = pd.DataFrame([exam_metrics])
exam_out = os.path.join(OUTPUT_DIR, SPLIT_METHOD, 'summary_metrics_exam_level.csv')
exam_metrics_df.to_csv(exam_out, index=False)
print(f'\nSaved: {exam_out}')


Exam-Level Metrics (pooled across all 10 GLOO folds):
---------------------------------------------
  AUC           : 0.7696
  Sensitivity   : 0.5455
  Specificity   : 0.8060
  PPV           : 0.4444
  NPV           : 0.8618
  Accuracy      : 0.7483
  TP            : 36
  TN            : 187
  FP            : 45
  FN            : 30

Saved: outputs\paper_revision\radiomics\09_final_pipeline\grouped_combinations_split_with_sampling\summary_metrics_exam_level.csv


## Animal-Level Results (A2)

Aggregate exam-level predictions to the mouse level using **majority vote** on `y_pred` across all scans for that mouse. Probability is averaged. This addresses Reviewer Comment A2: report animal-level performance in addition to exam-level.

With 10 mice (5 cured, 5 relapsing), a correct animal-level result is 10/10 mice classified correctly.


In [7]:
# Build exam-level results DataFrame
results_df = pd.DataFrame({
    'mouse_id': all_mouse_ids,
    'y_true': all_y_true,
    'y_pred': all_y_pred,
    'y_proba': all_y_proba,
})

# Majority vote per mouse (A2)
animal_df = results_df.groupby('mouse_id').agg(
    y_true=('y_true', 'first'),
    y_pred_vote=('y_pred', lambda x: int(x.mode()[0])),
    y_proba_mean=('y_proba', 'mean'),
).reset_index()

print('Animal-Level Predictions (majority vote across exams):')
print(animal_df.to_string(index=False))
print()

# Animal-level metrics
if len(animal_df['y_true'].unique()) > 1:
    animal_metrics = compute_metrics(
        animal_df['y_true'].values,
        animal_df['y_pred_vote'].values,
        animal_df['y_proba_mean'].values,
    )
else:
    animal_metrics = {k: float('nan') for k in
                      ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Accuracy',
                       'TP', 'TN', 'FP', 'FN']}

print('Animal-Level Metrics (N=10 mice):')
print('-' * 40)
for k, v in animal_metrics.items():
    if isinstance(v, float):
        print(f'  {k:<14}: {v:.4f}')
    else:
        print(f'  {k:<14}: {v}')

# Save
animal_df.to_csv(
    os.path.join(OUTPUT_DIR, SPLIT_METHOD, 'summary_metrics_animal_level.csv'), index=False
)
pd.DataFrame([animal_metrics]).to_csv(
    os.path.join(OUTPUT_DIR, SPLIT_METHOD, 'animal_level_metrics.csv'), index=False
)
print(f'\nSaved: summary_metrics_animal_level.csv, animal_level_metrics.csv')


Animal-Level Predictions (majority vote across exams):
 mouse_id  y_true  y_pred_vote  y_proba_mean
     1263       0            0      0.419466
     1264       0            0      0.366561
     1270       0            0      0.297729
     1276       1            1      0.531016
     1281       1            1      0.636627
     1284       1            1      0.658117
     1285       1            0      0.415292
     1380       0            0      0.306870
     1382       1            0      0.413509
     1383       0            0      0.399954

Animal-Level Metrics (N=10 mice):
----------------------------------------
  AUC           : 0.9200
  Sensitivity   : 0.6000
  Specificity   : 1.0000
  PPV           : 1.0000
  NPV           : 0.7143
  Accuracy      : 0.8000
  TP            : 3
  TN            : 5
  FP            : 0
  FN            : 2

Saved: summary_metrics_animal_level.csv, animal_level_metrics.csv


## Animal-Level Results — Day-Weighted Vote

Alternative to majority vote: each exam's predicted probability is **weighted by its
day of study** before aggregating per animal. Later exams receive higher weight because
the temporal analysis shows discrimination improves progressively (early AUC=0.576 vs
late AUC=0.962).

Two weighting schemes compared:
- **Linear:** weight = day_of_study (proportional to time)
- **Late-bias:** weight = day_of_study² (quadratic, penalises early exams more)

Animal prediction: `weighted_mean(y_proba) > 0.5 → cured`


In [8]:
# Day-weighted animal-level aggregation
# Uses pooled predictions already in memory (all_y_true, all_y_proba, all_mouse_ids, all_days)

weighted_rows = []
for mouse in np.unique(all_mouse_ids):
    mask = all_mouse_ids == mouse
    days    = all_days[mask].astype(float)
    proba   = all_y_proba[mask]
    y_true  = all_y_true[mask][0]  # same for all exams of this mouse

    # Linear weighting: weight proportional to day
    w_lin = days / days.sum()
    prob_lin = float(np.dot(w_lin, proba))

    # Quadratic weighting: weight proportional to day^2 (stronger late-bias)
    w_quad = days**2 / (days**2).sum()
    prob_quad = float(np.dot(w_quad, proba))

    weighted_rows.append({
        "mouse_id": mouse,
        "y_true": int(y_true),
        "n_exams": int(mask.sum()),
        "day_range": f"{int(days.min())}–{int(days.max())}",
        # Uniform (majority vote baseline)
        "prob_uniform": float(proba.mean()),
        "pred_uniform": int(proba.mean() > 0.5),
        # Linear weighting
        "prob_linear": prob_lin,
        "pred_linear": int(prob_lin > 0.5),
        # Quadratic weighting
        "prob_quad": prob_quad,
        "pred_quad": int(prob_quad > 0.5),
    })

wv_df = pd.DataFrame(weighted_rows)
print("Per-mouse predictions (uniform vs linear vs quadratic weighting):")
print(wv_df.to_string(index=False))
print()

# Compute animal-level metrics for each scheme
for scheme, pred_col, prob_col in [
    ("Uniform (majority vote)", "pred_uniform", "prob_uniform"),
    ("Linear day-weight",       "pred_linear",  "prob_linear"),
    ("Quadratic day-weight",    "pred_quad",    "prob_quad"),
]:
    m = compute_metrics(
        wv_df["y_true"].values,
        wv_df[pred_col].values,
        wv_df[prob_col].values,
    )
    print(f"{scheme}:")
    print(f"  AUC={m['AUC']:.4f}  Sens={m['Sensitivity']:.4f}  "
          f"Spec={m['Specificity']:.4f}  PPV={m['PPV']:.4f}  "
          f"Acc={m['Accuracy']:.4f}  TP={m['TP']} FP={m['FP']} FN={m['FN']} TN={m['TN']}")
    print()

# Save weighted vote results
wv_df.to_csv(
    os.path.join(OUTPUT_DIR, SPLIT_METHOD, "animal_level_weighted_vote.csv"),
    index=False
)
print("Saved: animal_level_weighted_vote.csv")


Per-mouse predictions (uniform vs linear vs quadratic weighting):
 mouse_id  y_true  n_exams day_range  prob_uniform  pred_uniform  prob_linear  pred_linear  prob_quad  pred_quad
     1263       0       56     11–47      0.419466             0     0.388206            0   0.362795          0
     1264       0       43     11–39      0.366561             0     0.355358            0   0.346582          0
     1270       0       43     10–37      0.297729             0     0.292948            0   0.288601          0
     1276       1       21     11–45      0.531016             1     0.558695            1   0.578413          1
     1281       1       12     10–27      0.636627             1     0.650411            1   0.661534          1
     1284       1        8     10–27      0.658117             1     0.679322            1   0.698018          1
     1285       1       17      9–31      0.415292             0     0.417646            0   0.424002          0
     1380       0       47    

## Temporal Analysis (A3)

Split all pooled predictions into **early / mid / late** terciles based on `day_of_study`, using the 33rd and 66th percentiles of the pooled day distribution as thresholds. Report AUC, Sensitivity, and Specificity for each temporal window.

This addresses Reviewer Comment A3: assess whether predictive performance is consistent across the treatment timeline.


In [9]:
# Compute tercile thresholds from pooled day_of_study distribution
t33, t66 = np.percentile(all_days, [33, 66])
print(f'Tercile thresholds: early <= {t33:.0f}, mid <= {t66:.0f}, late > {t66:.0f} (days)')
print()

temporal_rows = []
for window_name, mask in [
    ('early', all_days <= t33),
    ('mid',   (all_days > t33) & (all_days <= t66)),
    ('late',  all_days > t66),
]:
    yt  = all_y_true[mask]
    yp  = all_y_pred[mask]
    ypr = all_y_proba[mask]
    n = int(mask.sum())

    if n == 0 or len(np.unique(yt)) < 2:
        row = {'window': window_name, 'n_exams': n,
               'day_threshold': f'<={t33:.0f}' if window_name == 'early' else
                                (f'<={t66:.0f}' if window_name == 'mid' else f'>{t66:.0f}'),
               'AUC': float('nan'), 'Sensitivity': float('nan'), 'Specificity': float('nan')}
    else:
        m = compute_metrics(yt, yp, ypr)
        row = {'window': window_name, 'n_exams': n,
               'day_threshold': f'<={t33:.0f}' if window_name == 'early' else
                                (f'<={t66:.0f}' if window_name == 'mid' else f'>{t66:.0f}'),
               'AUC': m['AUC'], 'Sensitivity': m['Sensitivity'], 'Specificity': m['Specificity']}

    temporal_rows.append(row)

temporal_df = pd.DataFrame(temporal_rows)
print('Temporal Analysis (early / mid / late terciles):')
print(temporal_df.to_string(index=False))

temporal_df.to_csv(
    os.path.join(OUTPUT_DIR, SPLIT_METHOD, 'temporal_analysis.csv'), index=False
)
print(f'\nSaved: temporal_analysis.csv')


Tercile thresholds: early <= 21, mid <= 31, late > 31 (days)

Temporal Analysis (early / mid / late terciles):
window  n_exams day_threshold      AUC  Sensitivity  Specificity
 early      101          <=21 0.575944     0.441176     0.731343
   mid      100          <=31 0.802667     0.560000     0.760000
  late       97           >31 0.961905     1.000000     0.900000

Saved: temporal_analysis.csv


## Bootstrap 95% Confidence Intervals (A4)

Compute 95% CIs for exam-level AUC, Sensitivity, Specificity, PPV, and Accuracy using **non-parametric bootstrap** with 10,000 resamples (sampling with replacement from the pooled exam-level predictions). Bootstrap samples where only one class is present are skipped.

This addresses Reviewer Comment A4: quantify uncertainty around reported point estimates.


In [10]:
n_exams = len(all_y_true)
boot_metrics = defaultdict(list)

np.random.seed(SEED)
for _ in range(N_BOOTSTRAP):
    idx = np.random.choice(n_exams, size=n_exams, replace=True)
    yt_b  = all_y_true[idx]
    yp_b  = all_y_pred[idx]
    ypr_b = all_y_proba[idx]

    # Skip bootstrap samples with only one class present
    if len(np.unique(yt_b)) < 2:
        continue

    m = compute_metrics(yt_b, yp_b, ypr_b)
    for key in ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Accuracy']:
        val = m[key]
        if not np.isnan(val):
            boot_metrics[key].append(val)

print(f'Bootstrap 95% CIs (N={N_BOOTSTRAP} resamples, exam level):')
print('-' * 60)

boot_rows = []
for metric in ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Accuracy']:
    vals = np.array(boot_metrics[metric])
    lo, hi = np.percentile(vals, [2.5, 97.5])
    point = exam_metrics[metric]
    n_valid = len(vals)
    boot_rows.append({
        'metric': metric,
        'point_estimate': point,
        'ci_lower_2_5': lo,
        'ci_upper_97_5': hi,
        'n_bootstrap_valid': n_valid,
    })
    print(f'  {metric:<14}: {point:.4f}  95% CI [{lo:.4f}, {hi:.4f}]  (n_valid={n_valid})')

boot_df = pd.DataFrame(boot_rows)
boot_df.to_csv(
    os.path.join(OUTPUT_DIR, SPLIT_METHOD, 'bootstrap_cis.csv'), index=False
)
print(f'\nSaved: bootstrap_cis.csv')
print(boot_df.to_string(index=False))


Bootstrap 95% CIs (N=10000 resamples, exam level):
------------------------------------------------------------
  AUC           : 0.7696  95% CI [0.7033, 0.8322]  (n_valid=10000)
  Sensitivity   : 0.5455  95% CI [0.4211, 0.6667]  (n_valid=10000)
  Specificity   : 0.8060  95% CI [0.7553, 0.8559]  (n_valid=10000)
  PPV           : 0.4444  95% CI [0.3375, 0.5541]  (n_valid=10000)
  NPV           : 0.8618  95% CI [0.8148, 0.9065]  (n_valid=10000)
  Accuracy      : 0.7483  95% CI [0.6980, 0.7987]  (n_valid=10000)

Saved: bootstrap_cis.csv
     metric  point_estimate  ci_lower_2_5  ci_upper_97_5  n_bootstrap_valid
        AUC        0.769592      0.703319       0.832219              10000
Sensitivity        0.545455      0.421053       0.666667              10000
Specificity        0.806034      0.755274       0.855857              10000
        PPV        0.444444      0.337500       0.554058              10000
        NPV        0.861751      0.814815       0.906542              10000
   A

## Feature Importances

Aggregate XGBoost feature importances across all GLOO folds. Each feature's mean importance is computed over the folds in which it was selected. Features selected in fewer folds have higher variance in their importance estimates.


In [11]:
feat_imp_rows = []
for feat, imps in feature_importance_accum.items():
    feat_imp_rows.append({
        'feature': feat,
        'mean_importance': float(np.mean(imps)),
        'std_importance': float(np.std(imps)),
        'n_folds_selected': len(imps),
    })

feat_imp_df = (
    pd.DataFrame(feat_imp_rows)
    .sort_values('mean_importance', ascending=False)
    .reset_index(drop=True)
)

print(f'Top 20 features by mean importance across GLOO folds (N unique={len(feat_imp_df)}):')
print(feat_imp_df.head(20).to_string(index=False))

feat_out = os.path.join(OUTPUT_DIR, SPLIT_METHOD, 'top_features.csv')
feat_imp_df.to_csv(feat_out, index=False)
print(f'\nSaved: {feat_out}')

print('\n' + '=' * 60)
print('All outputs saved to:', os.path.join(OUTPUT_DIR, SPLIT_METHOD))
print('  - fold_*/predictions.csv  (per-fold exam predictions)')
print('  - summary_metrics_exam_level.csv')
print('  - summary_metrics_animal_level.csv')
print('  - animal_level_metrics.csv')
print('  - temporal_analysis.csv')
print('  - bootstrap_cis.csv')
print('  - top_features.csv')


Top 20 features by mean importance across GLOO folds (N unique=29):
                                        feature  mean_importance  std_importance  n_folds_selected
                    original_shape2D_Elongation         0.153986        0.024142                 9
original_gldm_DependenceNonUniformityNormalized         0.110420        0.026087                10
    original_glrlm_LongRunHighGrayLevelEmphasis         0.095251        0.017903                10
                      original_glrlm_RunEntropy         0.061173        0.014762                10
               original_shape2D_MajorAxisLength         0.048961        0.008480                10
               original_firstorder_90Percentile         0.048736        0.012824                10
               original_firstorder_10Percentile         0.047343        0.005676                10
                  original_glcm_Autocorrelation         0.044042        0.000000                 1
                    original_firstorder_M